# 百创S系列空间转录组数据生成Seurat对象
## 概述
百创S系列空间转录组矩阵图像信息与10X空间矩阵不同，不方便直接使用Seurat空间作图函数进行作图，根据S系列空间矩阵创建10x空间seurat对象格式，使用该方法创建object对象，可使用SpatialFeaturePlot等seurat内置空间作图函数来进行分析。  
随着百创S系列产品的升级，空间转录组数据的种类也在增多，百创S系列的空间转录组数据类型包括S1000，S2000，S3000, S4000, S5000以及细胞分割（cell_split）的数据，CreateBmkObject_v2.2.R适应更多的数据类型，具体用法如下：


In [ ]:

source("CreateBmkObject_v2.2.R")
object <- CreateBmkObject(matrix_path="07.CellSplit/mtx/", #矩阵文件目录
                          png_path="05.AllheStat/BSTViewer_project/he_roi_small.png", #png格式图片路径
                          min.cells = 5, #一个基因至少在n个细胞中表达才被保留，可自行调整，默认值5
                          min.features = 100, #一个细胞至少有n个基因才被保留，可自行调整，默认值100
                          spot_radius=0.005, #作图时点的半径
                          type="S3000", #数据类型，必须指定，可选参数有'S1000', 'S2000A', 'S2000B', 'S3000', 'S4000', 'S4000B', 'S5000'
                          cell_split=True, #是否为细胞分割数据
                          key="YF-SN4-CS3-0610-0611" #指定该样本对应的key
           ) 


## CreateBmkObject_v2.2.R下载
通过网盘分享的文件：CreateBmkObject_v2.2.R
链接: https://pan.baidu.com/s/1DWek-1ghqXo9DCwneIBXiw 提取码: hxq8


## 完整代码

In [ ]:
library(Seurat)
library(dplyr)
CreateBmkObject <- function(
  matrix_path,
  png_path,
  spot_radius = NULL,
  min.cells = 5,
  min.features = 100,
  type = NULL,
  cell_split = FALSE,
  key = "sample1"
  ){
  if(is.null(type)){
    stop("If the type parameter is empty, specify the type parameter. The options are 'S1000', 'S2000A', 'S2000B', 'S3000', 'S4000', 'S4000B', 'S5000'")
  }
  expr <- Seurat::Read10X(matrix_path, cell.column = 1)
  object <- Seurat::CreateSeuratObject(counts = expr,
                               assay = 'Spatial',
                               min.cells=min.cells,
                               min.features=min.features)
  #Image zoom rate
  cal_zoom_rate <- function(width, height, type){
    std_width = 1000
    if(type == "S1000" || type == "S2000A" || type == "S2000B" || type == "S3000" || type == "S4000" || type == "S4000B" || type == "S5000"){
      std_width = 1000
    }
    if(cell_split){
      std_width = 20000
    }
    if(type == "S3000"){
      std_height = std_width / (42 * 46) * (43 * 52 * sqrt(3) / 2.0)
    }else if(type == "S2000A"){
      std_height = std_width / (76 * 31) * (75 * 36 * sqrt(3) / 2.0)
    }else if(type == "S2000B"){
      std_height = std_width / (101 * 31) * (134 * 36 * sqrt(3) / 2.0)
    }else if(type == "S4000"){
      std_height = std_width / (76 * 46) * (75 * 52 * sqrt(3) / 2.0)
      std_height = std_width / (block_width * (spot_width+1)) * (block_height * (spot_height+1)* sqrt(3) / 2.0)
    }else if(type == "S4000B"){
      std_height = std_width / (101 * 46) * (134 * 52 * sqrt(3) / 2.0)
    }else if(type == "S5000"){
      std_height = std_width / (310 * 46) * (380 * 52 * sqrt(3) / 2.0)

    }else{
      std_height = std_width / (46 * 31) * (46 * 36 * sqrt(3) / 2.0)
    }
    if(std_width / std_height > width / height){
      scale = width / std_width
    }
    else{
      scale = height / std_height
    }
    return(scale)
  }
  #read png
  png <- png::readPNG(png_path)
  zoom_scale <-  cal_zoom_rate(dim(png)[2], dim(png)[1], type)
  #read barcode pos file
  ReadBarcodePos <- function(barcode_pos_path){
    barcode_pos <- read.table(gzfile(barcode_pos_path),header = F) %>%
      dplyr::rename(Barcode = V1 , pos_w = V2, pos_h = V3)
    return(barcode_pos)
  }
  #get barcode pos file path
  barcode_pos_path <- paste0(matrix_path,'/barcodes_pos.tsv.gz')
  barcode_pos <- ReadBarcodePos(barcode_pos_path = barcode_pos_path)
  if(!is.null(names(matrix_path))){
    barcode_pos$Barcode = paste(names(matrix_path), barcode_pos$Barcode, sep = "_")
  }
  barcode_pos <- barcode_pos %>% dplyr::filter(., Barcode %in% rownames(object@meta.data))
  #make spatial coord file for seurat S4 class
  coord <- data.frame(tissue = 1,
                      row = barcode_pos$pos_h,
                      col = barcode_pos$pos_w,
                      imagerow = barcode_pos$pos_h,
                      imagecol = barcode_pos$pos_w)
  rownames(coord) <- barcode_pos$Barcode
  #spot radius
  if(type == "S1000" || type == "S2000A" || type =='S2000B'){
    spot_radius_lib <- c(0.00063, 0.00179, 0.0027, 0.0039, 0.004, 0.0045, 0.005, NA, NA, NA, NA, NA, 0.0120)
  }else if(type == "S3000" || type == "S4000" || type == "S4000B" || type == "S5000"){
    spot_radius_lib <- c(0.00015, 0.00075, 0.0018, 0.0026, 0.003, 0.0039, 0.004, NA, 0.005, NA, NA, NA, NA, NA, NA, NA, 0.0120, 0.0120)
  }
  if(is.null(spot_radius)){
    spot_radius <- spot_radius_lib[as.numeric(gsub('L', '', strsplit(tail(strsplit(matrix_path, '/')[[1]],1), '_')[[1]][1]))]
  }else{
    spot_radius = spot_radius
  }
  if(is.null(spot_radius)){
    stop("The spot_radius parameter is null. Please specify the spot_radius parameter!!!")
  }
  #object
  sample1 <-  new(Class = "VisiumV1",
                  image = png,
                  scale.factors = Seurat::scalefactors(zoom_scale, 100, zoom_scale, zoom_scale),
                  coordinates = coord,
                  spot.radius = spot_radius,
                  assay = 'Spatial',
                  key = paste(key,"_",sep=""))
  #object@images <- list(key = sample1)
  object@images <- list()
  object@images[[key]] <- sample1
  return(object)
}
